## Projeto Sprint 8 - Análise de Negócio

### Contexto do Projeto
Trabalhamos no departamento analítico da Y.Afisha e fomos encarregados de otimizar os investimentos em marketing da empresa. Para isso, precisamos entender melhor o perfil dos consumidores e a eficiência dos diferentes canais de aquisição.

### Objetivo
Analisar os dados de janeiro de 2017 a dezembro de 2018 para otimizar os gastos com marketing e maximizar o retorno sobre investimento (ROI).

### Perguntas de Negócio
Este projeto busca responder às seguintes questões estratégicas:
* Qual é o perfil do nosso consumidor?
* Por quais canais de marketing nossos clientes chegam até nós?
* Quando os usuários começam a fazer compras após o primeiro acesso?
* Qual é o retorno sobre investimento (ROI) dos nossos canais de marketing?
* Quanto tempo demora para recuperar o investimento em aquisição de clientes?
* Qual canal oferece o melhor custo-benefício para futuros investimentos?

In [1]:
# Bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns   
import numpy as np



In [2]:
#Importando os dados de visitas(visits) já com a conversão nas colunas referentes a Start Ts e End Ts (object -> datetime).
# Convertendo a coluna Device para 'category' visto que só temos 2 categorias (Touch e Desktop) para que futuramente possa ser utilizada de forma mais eficiente.
#Além disso, convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
visits = pd.read_csv('visits_log_us.csv', 
                     parse_dates=['Start Ts', 'End Ts'], 
                     dtype={'Device': 'category'})
visits.columns = visits.columns.str.lower()

In [3]:
#importando os dados de pedidos (order) já com a conversão na coluna Buy Ts (object -> datetime).
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
order = pd.read_csv('orders_log_us.csv',
                     parse_dates=['Buy Ts'])
order.columns = order.columns.str.lower()

In [4]:
#Importando os dados de custos(costs) já com a conversão na coluna Dt de (object -> datetime).
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
#Convertendo os nomes das colunas para letras minúsculas para facilitar a manipulação dos dados.
costs = pd.read_csv('costs_us.csv',
                     parse_dates=['dt'])
costs.columns = costs.columns.str.lower()

## Análisando os DataFrames 

In [5]:
#Analisando os dados de visitas.
visits.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 359400 entries, 0 to 359399
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype         
---  ------     --------------   -----         
 0   device     359400 non-null  category      
 1   end ts     359400 non-null  datetime64[ns]
 2   source id  359400 non-null  int64         
 3   start ts   359400 non-null  datetime64[ns]
 4   uid        359400 non-null  uint64        
dtypes: category(1), datetime64[ns](2), int64(1), uint64(1)
memory usage: 11.3 MB


In [6]:
#Observando os dados de visitas(visits).
visits.head()

,device,end ts,source id,start ts,uid
0,touch,2017-12-20 17:38:00,4,2017-12-20 17:20:00,16879256277535980062
1,desktop,2018-02-19 17:21:00,2,2018-02-19 16:53:00,104060357244891740
2,touch,2017-07-01 01:54:00,5,2017-07-01 01:54:00,7459035603376831527
3,desktop,2018-05-20 11:23:00,9,2018-05-20 10:59:00,16174680259334210214
4,desktop,2017-12-27 14:06:00,3,2017-12-27 14:06:00,9969694820036681168


In [7]:
#Analisando os dados de pedidos(order).
order.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50415 entries, 0 to 50414
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype         
---  ------   --------------  -----         
 0   buy ts   50415 non-null  datetime64[ns]
 1   revenue  50415 non-null  float64       
 2   uid      50415 non-null  uint64        
dtypes: datetime64[ns](1), float64(1), uint64(1)
memory usage: 1.2 MB


In [8]:
#Observando os dados de vendas(order).
order.head()

,buy ts,revenue,uid
0,2017-06-01 00:10:00,17.00,10329302124590727494
1,2017-06-01 00:25:00,0.55,11627257723692907447
2,2017-06-01 00:27:00,0.37,17903680561304213844
3,2017-06-01 00:29:00,0.55,16109239769442553005
4,2017-06-01 07:58:00,0.37,14200605875248379450


In [9]:
#Analisando os dados de custos(costs).
costs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2542 entries, 0 to 2541
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   source_id  2542 non-null   int64         
 1   dt         2542 non-null   datetime64[ns]
 2   costs      2542 non-null   float64       
dtypes: datetime64[ns](1), float64(1), int64(1)
memory usage: 59.7 KB


In [10]:
costs.head()

,source_id,dt,costs
0,1,2017-06-01,75.20
1,1,2017-06-02,62.25
2,1,2017-06-03,36.53
3,1,2017-06-04,55.00
4,1,2017-06-05,57.08


### Análisando Coortes Comportamentais - Visits

In [40]:
#Criando as colunas date que representa o dia que o usuário acessou.
visits['date'] = visits['start ts'].dt.date
#Criando as colunas week que representa a semana que o usuário acessou, onde 0 representa segunda-feira e 6 representa domingo.
visits['week'] = visits['start ts'].dt.isocalendar().week
#Criando as colunas month que representa o mês que o usuário acessou.
visits['month'] = visits['start ts'].dt.to_period('M')    

In [56]:
#Contando Quantas pessoas usam-no cada dia, semana e mês.
unique_users_per_day = visits.groupby('date')['uid'].nunique().reset_index(name='unique_users') 
unique_users_per_week = visits.groupby('week')['uid'].nunique().reset_index(name='unique_users')
unique_users_per_month = visits.groupby('month')['uid'].nunique().reset_index(name='unique_users')


In [81]:
#Contando quantas dessas sessoes ocorreram em cada dia, agrupando por date e contando o número de sessões (uid) para cada dia.
sessions_per_day = visits.groupby('date')['uid'].count().reset_index(name='total_sessions')

In [79]:
#Usuários únicos por dia, semana e mês.
dau = visits.groupby('date')['uid'].nunique().mean()   
mau = visits.groupby('month')['uid'].nunique().mean()
wau = visits.groupby('week')['uid'].nunique().mean()

#Exibindo os resultados de DAU, MAU e WAU.
print(f'DAU médio: {dau:.2f}')
print(f'MAU médio: {mau:.2f}')
print(f'WAU médio: {wau:.2f}')

DAU médio: 907.99
MAU médio: 23228.42
WAU médio: 5825.29


In [48]:
#Calculando a duração de cada sessão em minutos, subtraindo o timestamp de início do timestamp de término e convertendo o resultado para minutos.
visits['duration_min'] = (
    visits['end ts'] - visits['start ts']
).dt.total_seconds() / 60

In [ ]:
#Retencao dos usuarios que voltaram no dia seguinte, 7 dias e 30 dias após a primeira visita.
# Data da primeira visita de cada usuário
first_visit = visits.groupby('uid')['date'].min().reset_index()
first_visit.columns = ['uid', 'first_date']

# Juntar ao dataset completo de visitas
visits = visits.merge(first_visit, on='uid')

#Calcula os dias desde a 1ª visita
visits['days_since_first'] = (
    pd.to_datetime(visits['date']) - pd.to_datetime(visits['first_date'])
).dt.days

# Conta os usuários que voltaram em cada janela. 
total = visits['uid'].nunique()
ret_1d  = visits[visits['days_since_first'] == 1]['uid'].nunique()
ret_7d  = visits[(visits['days_since_first'] > 0) & 
                 (visits['days_since_first'] <= 7)]['uid'].nunique()
ret_30d = visits[(visits['days_since_first'] > 0) & 
                 (visits['days_since_first'] <= 30)]['uid'].nunique()

# Exibir as taxas de retenção.
print(f"Dia 1: {ret_1d/total*100:.1f}% | 7d: {ret_7d/total*100:.1f}% | 30d: {ret_30d/total*100:.1f}%")

Dia 1: 2.6% | 7d: 6.4% | 30d: 11.0%
